# Roman Urdu captions — Colab training

**Runtime → Change runtime type → T4 GPU** before running anything.

Then **Runtime → Run all**. Every cell is idempotent and the whole thing is
resumable, so if the session drops you re-open and run all again — it picks up
from the last checkpoint on Drive rather than starting over.

Free Colab disconnects at roughly four hours, so training is stopped by a clock
(`MAX_TRAIN_MINUTES`), not by an epoch. An unfinished run that saved nothing is
worth zero; a partial adapter is worth continuing.

**Never paste a token into a cell.** A `.ipynb` stores its output, so a printed
token is committed with the file. Use the key icon in the left sidebar to add
`HF_TOKEN` as a Colab secret.

In [ ]:
# --- Settings + logging ---------------------------------------------------
import os
import shutil
import subprocess
import time
from datetime import datetime

REPO = "https://github.com/nabeeltahirdeveloper/STT-Model.git"
BRANCH = "phase0-baseline-and-spelling-spec"
HF_MODEL = "MubeenAmjad205/roman-urdu-captions"  # private
TRAIN_HOURS = 20  # audio hours to fetch (~0.62 GB each)
MAX_TRAIN_MINUTES = 150  # leaves room for download, eval and upload in 4 h

# Everything printed also lands here. Colab clears cell output on reconnect and
# truncates long streams, so the file is the copy that survives -- download it
# from the file browser if a run needs diagnosing.
LOG = "/content/session.log"
T0 = time.time()


def elapsed():
    return f"{(time.time() - T0) / 60:.0f} min elapsed"


def log(line=""):
    print(line, flush=True)
    with open(LOG, "a", encoding="utf-8") as fh:
        fh.write(f"{line}\n")


def step(title):
    log("")
    log("=" * 78)
    log(f"[{datetime.now():%H:%M:%S} | {elapsed()}] {title}")
    log("=" * 78)


def sh(cmd, check=True, tail=40):
    """Run a command, streaming its output to the cell and to LOG.

    subprocess.run(check=True) raises CalledProcessError, and Colab renders that
    traceback so prominently that the real error above it scrolls out of view --
    which is how an exit 1 arrives looking like it has no cause. Here output is
    captured as it streams and a failure reprints the tail under a banner that
    cannot be missed, so the reason is always the last thing on screen.
    """
    log(f"$ {cmd}")
    started = time.time()
    lines = []
    proc = subprocess.Popen(
        cmd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for raw in proc.stdout:
        raw = raw.rstrip()
        lines.append(raw)
        log(raw)
    code = proc.wait()
    log(f"[exit {code} | {time.time() - started:.0f}s]")
    if code != 0:
        log("")
        log("!" * 78)
        log(f"!! FAILED (exit {code}): {cmd}")
        log(f"!! last {min(tail, len(lines))} lines of its output:")
        log("!" * 78)
        for raw in lines[-tail:]:
            log(f"  | {raw}")
        log("!" * 78)
        if check:
            raise RuntimeError(f"see the banner above -- {cmd}")
    return code


def gpu(note=""):
    import torch

    if not torch.cuda.is_available():
        log("    no GPU")
        return
    free, total = torch.cuda.mem_get_info()
    log(
        f"    GPU {(total - free) / 1e9:.1f}/{total / 1e9:.1f} GB in use"
        f" | peak allocated {torch.cuda.max_memory_allocated() / 1e9:.1f} GB {note}"
    )


def disk():
    used = shutil.disk_usage("/content")
    log(
        f"    disk {used.used / 1e9:.0f}/{used.total / 1e9:.0f} GB used,"
        f" {used.free / 1e9:.0f} GB free"
    )


open(LOG, "w").close()
step("Session start")
log(f"log file: {LOG}")
disk()

In [ ]:
# --- GPU check ------------------------------------------------------------
step("GPU check")

# Fail here rather than 40 minutes in. A CPU runtime will "work" and take days.

import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again."
)

name = torch.cuda.get_device_name(0)

vram = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"{name} · {vram:.0f} GB VRAM")

In [ ]:
# --- Where checkpoints live ------------------------------------------------
step("Checkpoint destinations")

# No Google Drive. /content is wiped when the session ends, so the private

# HuggingFace repo is the durable copy -- and it is offsite, which Drive on the

# same account is not. train pushes there every `push_every` steps and on the

# time-box, so a disconnect costs minutes rather than the run.

OUT = "/content/finetuned"

os.makedirs(OUT, exist_ok=True)

print("local:", OUT, "· durable:", HF_MODEL)

In [ ]:
# --- Repo + dependencies --------------------------------------------------
step("Repo and dependencies")

if not os.path.isdir("/content/model"):
    sh(f"git clone --branch {BRANCH} --single-branch {REPO} /content/model")

os.chdir("/content/model")

sh("git pull --ff-only", check=False)


sh("pip -q install uv")

# The `train` extra carries bitsandbytes and accelerate. They are NOT installed

# with `uv pip install`: uv run re-syncs the environment and prunes whatever is

# undeclared, so the package disappears before finetune.py imports it.

#

# `--extra cuda` is deliberately absent. That extra is flash-attn, which is

# marked no-build-isolation and so needs torch installed *before* it builds --

# it cannot come from the same sync. It is a speed optimization the T4 run does

# not need.

sh("uv sync --extra train")


# Fail in the setup cell, not 40 minutes into the session.

sh(
    "uv run python -c 'import bitsandbytes, torch; "
    'print("bitsandbytes", bitsandbytes.__version__, "| torch", torch.__version__)\''
)

print(elapsed())

In [ ]:
# --- Diagnostics: the state everything else depends on --------------------
# Written down before training rather than after a failure, because half of
# what matters here (commit sha, package versions, free disk) is not
# recoverable once the runtime is gone.
step("Environment")

import sys

log(f"python   {sys.version.split()[0]}")
log(f"torch    {torch.__version__} | cuda {torch.version.cuda}")
log(f"gpu      {torch.cuda.get_device_name(0)}")
gpu("(baseline, before any model is loaded)")
disk()

sh("git -C /content/model log --oneline -1", check=False)
sh("git -C /content/model status --short", check=False)
sh(
    'uv run python -c "import bitsandbytes, transformers, peft;'
    " print('bitsandbytes', bitsandbytes.__version__);"
    " print('transformers', transformers.__version__);"
    " print('peft', peft.__version__)\"",
    check=False,
)
sh(
    "nvidia-smi --query-gpu=name,memory.used,memory.total,temperature.gpu --format=csv,noheader",
    check=False,
)

In [ ]:
# --- Auth: from Colab secrets, never from a cell --------------------------
step("HuggingFace auth")

from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

    from huggingface_hub import whoami

    print("hugging face:", whoami()["name"])

except Exception as e:
    print("No HF_TOKEN secret — training will run, upload will be skipped.")

    print("Add it with the key icon in the left sidebar.", type(e).__name__)

In [ ]:
# --- Data -----------------------------------------------------------------
step("Data: transcripts, labels, subset, audio")

from pathlib import Path

# Transcripts are tiny; audio is the slow part. Both download inside Google's

# network, far faster than a home connection, and both are resumable.

sh(
    "uv run python -m scripts.download_transcripts --corpus-transcripts",
    check=False,
)


# data/labels/ is gitignored, so the clone has no labels -- they are generated

# rather than shipped. Dictionary lookup makes that ~3 seconds for 29,749

# utterances, which is why they are not worth versioning (ADR-014).

sh("uv run python -m scripts.build_labels --split US-CS")


sh(f"uv run python -m scripts.select_training_subset --hours {TRAIN_HOURS}")

sh(
    "uv run python -m scripts.fetch_training_audio --manifest data/labels/train-subset.jsonl",
    check=False,
)

log("\nwhat the data steps produced:")
for path in ("data/labels/labels.jsonl", "data/labels/train-subset.jsonl"):
    if os.path.exists(path):
        n = len(Path(path).read_text(encoding="utf-8").splitlines())
        log(f"    {path}: {n:,} rows")
    else:
        log(f"    {path}: MISSING")
disk()
log(elapsed())

In [ ]:
# --- Preflight: will training find anything to train on? ------------------
# finetune.py exits non-zero when no manifest row survives its filter, and the
# filter is stricter than "the file exists": the audio must be on disk, the
# duration must be 1-30 s, and the label must be non-empty. A silent download
# failure therefore surfaces as an unexplained exit 1 from the training script,
# several cells away from the cell that actually went wrong.
#
# This reproduces that filter and reports which condition removed each row, so
# the diagnosis arrives before the session is spent rather than after.
step("Preflight: manifest and audio")

import json
from pathlib import Path

MANIFEST = Path("data/labels/train-subset.jsonl")
if not MANIFEST.exists():
    raise RuntimeError(f"{MANIFEST} does not exist -- the data cell did not finish")

lines = MANIFEST.read_text(encoding="utf-8").splitlines()
rows = [json.loads(line) for line in lines if line.strip()]
usable, no_audio, bad_duration, no_label = 0, 0, 0, 0
missing_examples = []
for row in rows:
    audio = Path(str(row["audio"]))
    if not audio.exists():
        no_audio += 1
        if len(missing_examples) < 3:
            missing_examples.append(str(audio))
        continue
    if not (1.0 <= float(row.get("duration_s") or 0) <= 30.0):
        bad_duration += 1
        continue
    if not str(row.get("label") or "").strip():
        no_label += 1
        continue
    usable += 1

log(f"manifest rows      {len(rows):,}")
log(f"  usable           {usable:,}   <- what training will actually see")
log(f"  audio missing    {no_audio:,}")
log(f"  duration out of range {bad_duration:,}")
log(f"  label empty      {no_label:,}")

audio_root = Path("data/raw/urduspeech")
on_disk = list(audio_root.rglob("*.wav")) if audio_root.exists() else []
log(f"audio files under data/raw/urduspeech: {len(on_disk):,}")
for example in missing_examples:
    log(f"  expected but absent: {example}")
disk()

if usable == 0:
    raise RuntimeError(
        "no usable rows -- training would exit 1 here. If 'audio missing' is the "
        "whole manifest, re-run the data cell: fetch_training_audio is resumable."
    )
log(f"\nOK: {usable:,} clips ready.")

In [ ]:
# --- Train: FULL fine-tune, not LoRA --------------------------------------
# LoRA on these labels scored CER 64.1% against 34.9% for the stock model plus
# our romanizer -- worse than not training. Full fine-tuning is the recipe
# ADR-003 specifies; 8-bit Adam plus gradient checkpointing is what makes
# ~780M trainable parameters fit in a 16 GB T4.
step("Smoke test: 60 clips")
gpu("(before smoke)")

# Verify on 60 clips before spending the session. This takes about a minute and
# would have caught every mistake made getting here.
sh(f"uv run python -m src.training.finetune --limit 60 --out {OUT}-smoke")
gpu("(after smoke -- this is the real memory ceiling)")

step("Full run")
resume = f"--resume {OUT}" if os.path.exists(f"{OUT}/config.json") else ""
remaining = max(15, MAX_TRAIN_MINUTES - (time.time() - T0) / 60)
log(f"resuming: {bool(resume)} | time-box: {remaining:.0f} min")
sh(
    f"uv run python -m src.training.finetune --out {OUT} {resume} "
    f"--max-minutes {remaining:.0f} --hf-repo {HF_MODEL} --push-every 300"
)
gpu("(after full run)")
disk()
log(elapsed())

In [ ]:
# --- The gate: CER, not script mix ----------------------------------------
step("Evaluation")

# This session's hard lesson. Script mix went 5.7% -> 100% Latin across three

# runs and was reported as success; CER had meanwhile gone 34.9% -> 64.1%. The

# model got twice as wrong while looking twice as good.

sh(
    "uv run python -m scripts.download_transcripts --corpus-transcripts",
    check=False,
)


sh(f"uv run python -m src.eval.score --pred {OUT}/predictions.txt", check=False)

print("\nCompare against 34.9% -- stock 0.6B plus our romanizer on the same clips.")

print("Above it means training did not help, whatever else looks good.")

In [ ]:
# --- Already uploaded ------------------------------------------------------
# train pushes to the private HF repo during the run, so there is nothing to do
# here unless the run was interrupted before its first push.
print("model ->", HF_MODEL, "(private)")
print(elapsed())

## If the session dies

Re-open this notebook and **Run all**. The repo is already cloned, the audio is
already downloaded, and training resumes from the Drive checkpoint. Nothing is
repeated that does not need to be.

## Reading the result

The only number that matters is **CER against the eval set**. Compare it with
stock 0.6B plus our romanizer, which scored **34.9%** on the same clips.

- **Below 34.9%** — training helped. That is the first real evidence it does.
- **Above** — it did not, whatever the script mix says. Lower the learning rate
  (`--learning-rate 2e-5`) and check the loss curve is falling rather than
  rising before spending another session on it.

Do not compare against the 27.9% baseline: that is the 1.7B model on the full
set, and ADR-016 explains why the comparison is not like for like.